In [11]:
from pymilvus import Collection, MilvusClient, connections, db, utility
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.runnables.schema import StreamEvent

from uuid import uuid4
import os
import re
import json
from _collections_abc import AsyncIterator
from typing import Tuple
import tiktoken
import pymupdf


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Lars\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Lars\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_i

In [12]:
client = MilvusClient(
    db_name='document_embeddings',
)

In [13]:
client.list_collections()

['LangChainCollection', 'response_cache']

In [14]:
client.describe_collection('LangChainCollection')

{'collection_name': 'LangChainCollection',
 'auto_id': False,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 100,
   'name': 'text',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535}},
  {'field_id': 101,
   'name': 'pk',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535},
   'is_primary': True},
  {'field_id': 102,
   'name': 'vector',
   'description': '',
   'type': <DataType.FLOAT_VECTOR: 101>,
   'params': {'dim': 3072}},
  {'field_id': 103,
   'name': 'Document',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535}},
  {'field_id': 104,
   'name': 'pages',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535}},
  {'field_id': 105,
   'name': 'page_labels',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535}}],
 'functions': [],
 'aliases': [],
 'collection_id': 461643

In [18]:
def new_cache_entry(prompt: str, response: str) -> None:
    db_uri = 'http://localhost:19530'
    vector_store = Milvus(
        collection_name='response_cache',
        embedding_function=OpenAIEmbeddings(model='text-embedding-3-large'),
        connection_args={
            'uri': db_uri,
            'token': 'root:Milvus',
            'db_name': 'document_embeddings'
        },
        index_params={
            'index_type': 'FLAT',
            'metric_type': 'COSINE',
        },
        consistency_level='Strong'
    )

    vector_store.add_documents(
        [Document(page_content=prompt, metadata={'response': response, 'prompt': prompt})],
        ids=[str(uuid4())]  # Generate unique IDs
    )

In [ ]:
def query_cache(prompt: str) -> str|None:
    db_uri = 'http://localhost:19530'
    vector_store = Milvus(
        collection_name='response_cache',
        embedding_function=OpenAIEmbeddings(model='text-embedding-3-large'),
        connection_args={
            'uri': db_uri,
            'token': 'root:Milvus',
            'db_name': 'document_embeddings'
        },
        index_params={
            'index_type': 'FLAT',
            'metric_type': 'COSINE',
        },
        consistency_level='Strong'
    )

    results = vector_store.as_retriever(
        search_type='similarity_score_threshold',
        search_kwargs={
            'k': 1,
            'score_threshold': 0.8
        }
    ).invoke(prompt)

    if results:
        return results
    
    return None

In [30]:
query_cache("test")

No relevant docs were retrieved using the relevance score threshold 0.8


In [24]:
new_cache_entry("What is the capital of France?", "The capital of France is Paris.")

In [1]:
from config.config import load_config
from dataclasses import asdict

In [2]:
config = load_config('../volumes/data/config.yaml')

In [3]:
config

Config(rag_config=RAGConfig(llm_models=['o4-mini', 'gpt-4.1', 'gpt-5', 'gpt-5-mini'], embedding_model='text-embedding-3-large', vector_db_name='document_embeddings', index_type='FLAT', metric_type='L2', search_type='similarity_score_threshold', search_kwargs={}))

In [1]:
from pydantic import BaseModel, Field

class Test(BaseModel):
    test: list[int]

    def __getitem__(self, index: int) -> int:
        return self.test[index]
    
    def __iter__(self):
        for item in self.test:
            yield item

In [2]:
test = Test(test=[1,2,3,4,5])

for i in test:
    print(i)

1
2
3
4
5


In [3]:
test = {
    'a': 'b',
    'b': 'c'
}

class TestModel(BaseModel):
    a: str

t = TestModel(**test)
t

TestModel(a='b')